# 10 — Modelos baseados em grafos (TEA): GCN, GAT e BrainGNN

**Modelos comparados:** GCN, GAT e BrainGNN. O BrainGNN usa convolução ROI-aware e seleção Top-K de ROIs. Nenhum deles é prometido como melhor que os baselines: com sinal fraco nos dados, o resultado honesto pode ficar perto do acaso.

**Decisões de construção do grafo (todas explícitas)**

| Decisão | Escolha | Observação |
|---|---|---|
| Unidade | **Um grafo por participante** | Sem grafo populacional com treino e teste juntos (evita vazamento por estrutura) |
| Nós | As 200 ROIs (inferidas do tamanho de `X`) | |
| Atributos do nó | Linha da matriz em **Fisher z, com sinal** (tamanho = nº de ROIs) | Padronizados com média/desvio **só do treino** do fold externo |
| Arestas | As `k%` maiores |r| do participante (10% ou 20%) | Densidade escolhida **só pela validação interna** |
| Correlações negativas | Entram pela magnitude |r|; o **sinal fica nos atributos do nó** | Alternativa (arestas assinadas) não testada |
| Diagonal | Removida; autoconexão adicionada só no passo de mensagem (GCN/GAT) | |
| Peso da aresta | |r| na propagação do GCN; atenção aprendida no GAT; pesos ROI-aware e R-pool no BrainGNN | |
| Classificação | Leitura do grafo (média + máximo dos nós) → MLP → 1 saída por participante | Em lotes de grafos densos (B × n × n) |
| Explicação | A atenção do GAT **não** é explicação biológica definitiva | Só hipótese exploratória (notebook 11) |

**Protocolo**
- Mesmos 5 folds externos por site do notebook 07; normalização só no treino; validação interna agrupada por site (a mesma checagem de viabilidade dos notebooks 08/09) para early stopping, **escolha da densidade (10% vs 20%)** e do limiar.
- Hiperparâmetros fixos e pequenos (sem grade grande): 2 camadas, 32 unidades, dropout 0,5, AdamW (lr 1e-3, wd 1e-3). Várias sementes. A tabela longa guarda tudo.
- Os folds são validação cruzada por site, não um teste independente.

A implementação BrainGNN é uma adaptação compacta com Ra-GConv e R-pool Top-K; não inclui as perdas unit, TPK e GLC do artigo. Os resultados antigos de GCN/GAT ficam preservados; os novos resultados desta versão são gravados em pasta separada.


In [1]:
# 1. Projeto, módulo de avaliação e dataset (TEA / ABIDE I)
import sys, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

_raiz = Path.cwd().resolve()
for _c in (_raiz, *_raiz.parents):
    if (_c / 'src' / 'config.py').exists():
        _raiz = _c
        break
else:
    raise RuntimeError('Não encontrei src/config.py. Abra a pasta TCC_TEA_TDAH no VS Code.')
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src import avaliacao as av
criar_pastas()

NOME_BASE = 'TEA'
ROTULO_CASO = 'TEA'
ARQ_DATASET = PATHS['processed'] / 'dataset_abide_processado.pkl'
PASTA_RES = BASE_DIR / 'experimentos' / '08_grafos_tea' / 'Results'
PASTA_MOD = BASE_DIR / 'experimentos' / '08_grafos_tea' / 'Models'
PASTA_RES.mkdir(parents=True, exist_ok=True)
PASTA_MOD.mkdir(parents=True, exist_ok=True)

# 2. Parâmetros desta execução
import copy
import torch, torch.nn as nn, torch.nn.functional as F

DENSIDADES = [0.10, 0.20]
SEMENTES = [42, 43, 44]
RODAR_GAT = True
RODAR_BRAINGNN = True         # ativar BrainGNN
HID, DROP, LR, WD = 32, 0.5, 1e-3, 1e-3
EPOCAS_MAX, PACIENCIA, BATCH, EPOCAS_FIXAS = 80, 12, 32, 30
if av.RAPIDO:
    DENSIDADES, SEMENTES, EPOCAS_MAX, PACIENCIA, EPOCAS_FIXAS = [0.10, 0.20], [42], 3, 2, 3
    print('*** TCC_RAPIDO=1: teste curto ***')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
d = av.carregar_dataset(ARQ_DATASET)
N = av.n_rois_de(d['X'].shape[1])
print(f'Base: {NOME_BASE} | participantes: {len(d["y"])} | ROIs (nós): {N} | dispositivo: {DEVICE}')
print('Classes:', pd.Series(d['y']).value_counts().sort_index().to_dict(), '| sites:', len(np.unique(d['sites'])))

_i, _j = np.triu_indices(N, k=1)
Z = np.zeros((len(d['X']), N, N), dtype=np.float32)       # Fisher z (com sinal), diagonal = 0
Z[:, _i, _j] = d['X']; Z[:, _j, _i] = d['X']
ABSR = np.abs(np.tanh(Z)).astype(np.float32)               # |r| (a diagonal já é 0)
EYE = np.eye(N, dtype=bool)
print('Z:', Z.shape, '| memória aproximada: %.0f MB' % (2 * Z.nbytes / 1e6))

def construir_arestas(dens):
    """Seleciona exatamente round(dens * 19.900) arestas não direcionadas por participante.

    A ordenação usa |Fisher Z| (mesma ordem de |Pearson|), desempate estável pela
    posição da conexão; a matriz final é simétrica e não contém autoconexões.
    """
    ai, aj = np.triu_indices(N, k=1)
    k = int(round(dens * len(ai)))
    A = np.zeros((len(d['X']), N, N), dtype=np.uint8)
    for row in range(len(d['X'])):
        ordem = np.argsort(-np.abs(d['X'][row]), kind='stable')[:k]
        ii, jj = ai[ordem], aj[ordem]
        A[row, ii, jj] = 1; A[row, jj, ii] = 1
    return A

ARESTAS = {dens: construir_arestas(dens) for dens in DENSIDADES}
for dens, A in ARESTAS.items():
    print(f'densidade {dens:.0%}: arestas por grafo = {int(A[0].sum() // 2)} (esperado ~{int(round(dens * N * (N - 1) / 2))}); simétrico: {bool((A == A.transpose(0, 2, 1)).all())}')

Base: TEA | participantes: 871 | ROIs (nós): 200 | dispositivo: cpu
Classes: {0: 468, 1: 403} | sites: 20
Z: (871, 200, 200) | memória aproximada: 279 MB
densidade 10%: arestas por grafo = 1990 (esperado ~1990); simétrico: True
densidade 20%: arestas por grafo = 3980 (esperado ~3980); simétrico: True


In [2]:
# 3. Camadas e modelos densos (um lote = B grafos de N nós)
class CamadaGCN(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o)
    def forward(self, H, A, W):
        # A_hat = D^-1/2 (A*|r| + I) D^-1/2
        At = W * A + torch.eye(H.shape[1], device=H.device)
        dinv = At.sum(-1).clamp(min=1e-6).pow(-0.5)
        return torch.bmm(dinv.unsqueeze(-1) * At * dinv.unsqueeze(-2), self.lin(H))

class CamadaGAT(nn.Module):
    def __init__(self, i, o, heads=4):
        super().__init__()
        self.h, self.o = heads, o
        self.W = nn.Linear(i, heads * o, bias=False)
        self.a_s = nn.Parameter(torch.randn(heads, o) * 0.1); self.a_d = nn.Parameter(torch.randn(heads, o) * 0.1)
        self.drop = nn.Dropout(0.3)
    def forward(self, H, A, W):
        B, n, _ = H.shape
        Hw = self.W(H).view(B, n, self.h, self.o)                                   # (B,n,h,o)
        es = (Hw * self.a_s).sum(-1); ed = (Hw * self.a_d).sum(-1)                  # (B,n,h)
        e = F.leaky_relu(es.permute(0, 2, 1).unsqueeze(-1) + ed.permute(0, 2, 1).unsqueeze(-2), 0.2)   # (B,h,n,n)
        self_loop = torch.eye(n, device=A.device).expand(B, -1, -1)
        mascara = (A + self_loop) > 0
        edge_weight = A * W + self_loop
        e = e + torch.log(edge_weight.clamp_min(1e-8)).unsqueeze(1)
        e = e.masked_fill(~mascara.unsqueeze(1), float('-inf'))
        att = self.drop(torch.softmax(e, dim=-1))
        out = torch.matmul(att, Hw.permute(0, 2, 1, 3))                             # (B,h,n,o)
        return out.permute(0, 2, 1, 3).reshape(B, n, self.h * self.o)

class CamadaRaGConv(nn.Module):
    """Convolução ROI-aware: transformação aprendida separadamente por ROI."""
    def __init__(self,n,i,o):
        super().__init__(); self.weight=nn.Parameter(torch.empty(n,i,o)); self.bias=nn.Parameter(torch.zeros(n,o)); nn.init.xavier_uniform_(self.weight)
    def forward(self,H,A,W):
        n=H.shape[1]; At=A*W+torch.eye(n,device=H.device); dinv=At.sum(-1).clamp(min=1e-6).pow(-.5)
        Ahat=dinv.unsqueeze(-1)*At*dinv.unsqueeze(-2); msg=torch.bmm(Ahat,H)
        return torch.einsum('bni,nio->bno',msg,self.weight)+self.bias

class RedeGrafo(nn.Module):
    def __init__(self, tipo, n_in, hid=HID, drop=DROP):
        super().__init__()
        self.tipo = tipo
        if tipo == 'GCN':
            self.l1, self.l2, d_out = CamadaGCN(n_in, hid), CamadaGCN(hid, hid), hid
        elif tipo == 'GAT':
            self.l1, self.l2 = CamadaGAT(n_in, hid // 4, 4), CamadaGAT(hid, hid // 4, 4); d_out = hid
        elif tipo == 'BrainGNN':
            self.l1, self.l2, d_out = CamadaRaGConv(N,n_in,hid), CamadaRaGConv(N,hid,hid), hid
            self.roi_score = nn.Linear(hid,1)
        else:
            raise ValueError(tipo)
        self.drop = nn.Dropout(drop)
        self.mlp = nn.Sequential(nn.Linear(2 * d_out, 32), nn.ReLU(), nn.Dropout(drop), nn.Linear(32, 1))
    def forward(self, Xf, A, W):
        H = self.drop(F.elu(self.l1(Xf, A, W)) if self.tipo == 'GAT' else F.relu(self.l1(Xf, A, W)))
        mask = None
        if self.tipo == 'BrainGNN':
            scores=self.roi_score(H).squeeze(-1); k=max(10,int(np.ceil(H.shape[1]*.5))); idx=scores.topk(k,dim=1).indices
            mask=torch.zeros_like(scores).scatter_(1,idx,1.0); H=H*torch.sigmoid(scores).unsqueeze(-1)*mask.unsqueeze(-1)
            keep=mask.unsqueeze(2)*mask.unsqueeze(1); A,W=A*keep,W*keep
        H = F.elu(self.l2(H, A, W)) if self.tipo == 'GAT' else F.relu(self.l2(H, A, W))
        if mask is None: leitura=torch.cat([H.mean(1),H.max(1).values],dim=1)
        else:
            media=(H*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp(min=1)
            maximo=H.masked_fill(mask.unsqueeze(-1)==0,float('-inf')).max(1).values
            leitura=torch.cat([media,maximo],dim=1)                    # nível do grafo (participante)
        return self.mlp(self.drop(leitura)).squeeze(1)

def lote(FEATS, dens, idx):
    return (torch.as_tensor(FEATS[idx]).to(DEVICE),
            torch.as_tensor(ARESTAS[dens][idx].astype(np.float32)).to(DEVICE),
            torch.as_tensor(ABSR[idx]).to(DEVICE))

@torch.no_grad()
def prever_g(modelo, FEATS, dens, idx, batch=BATCH):
    modelo.eval(); out = []
    for k in range(0, len(idx), batch):
        out.append(torch.sigmoid(modelo(*lote(FEATS, dens, idx[k:k + batch]))).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def treinar_g(modelo, FEATS, dens, y, idx_tr, idx_va, semente):
    g = torch.Generator().manual_seed(semente)
    modelo = modelo.to(DEVICE)
    n_pos, n_neg = int((y[idx_tr] == 1).sum()), int((y[idx_tr] == 0).sum())
    perda_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE))
    opt = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=WD)
    tem_val = idx_va is not None and len(np.unique(y[idx_va])) == 2
    epocas = EPOCAS_MAX if tem_val else EPOCAS_FIXAS
    melhor, estado, sem_melhora, rodadas, melhor_ep = -1.0, None, 0, 0, 0
    for ep in range(epocas):
        modelo.train(); rodadas += 1
        ordem = idx_tr[torch.randperm(len(idx_tr), generator=g).numpy()]
        for k in range(0, len(ordem), BATCH):
            b = ordem[k:k + BATCH]
            if len(b) < 2: continue
            opt.zero_grad(set_to_none=True)
            perda = perda_fn(modelo(*lote(FEATS, dens, b)), torch.as_tensor(y[b], dtype=torch.float32, device=DEVICE))
            perda.backward(); opt.step()
        if tem_val:
            auc = av.roc_auc_score(y[idx_va], prever_g(modelo, FEATS, dens, idx_va))
            if auc > melhor + 1e-4: melhor, estado, sem_melhora, melhor_ep = auc, copy.deepcopy(modelo.state_dict()), 0, ep + 1
            else: sem_melhora += 1
            if sem_melhora >= PACIENCIA: break
    if tem_val and estado is not None: modelo.load_state_dict(estado)
    return modelo, (melhor if tem_val else np.nan), (melhor_ep if tem_val else rodadas)

def limiar_interno(y_va, p_va):
    if y_va is None or len(np.unique(y_va)) < 2: return 0.5
    cand = np.unique(np.round(p_va, 4))
    return float(max((av.balanced_accuracy_score(y_va, (p_va >= t).astype(int)), -abs(t - 0.5), t) for t in cand)[2])

def normalizar_atributos(idx_tr):
    """Média/desvio por coluna do atributo, só com participantes e nós do treino."""
    v = Z[idx_tr].reshape(-1, N)
    mu, sd = v.mean(0), v.std(0) + 1e-6
    return ((Z - mu) / sd).astype(np.float32)

In [3]:
# 4. Validação externa agrupada e validação interna completa
print('FOLDS EXTERNOS'); display(av.resumo_folds(d))
tipos = ['GCN'] + (['GAT'] if RODAR_GAT else []) + (['BrainGNN'] if RODAR_BRAINGNN else [])
folds = sorted(np.unique(d['fold_id']).tolist())
proba_oof = {m: np.full(len(d['y']), np.nan, dtype=np.float32) for m in tipos}
pred_oof = {m: np.full(len(d['y']), -1, dtype=np.int8) for m in tipos}
linhas, escolhas = [], []
(PASTA_MOD / 'grafos_folds').mkdir(exist_ok=True)

for fold in folds:
    te = np.where(d['fold_id'] == fold)[0]; tr = np.where(d['fold_id'] != fold)[0]
    splits, info = av.splits_internos(d['y'][tr], d['sites'][tr], k_max=3, seed=SEED)
    if not info['viavel']:
        warnings.warn(f'Fold {fold}: sem partição interna agrupada viável; fold registrado sem treino externo neste protocolo.')
        continue
    splits_abs = [(tr[a], tr[b]) for a, b in splits]
    FEATS = normalizar_atributos(tr)  # estatísticas apenas no treino externo
    print(f"\n=== Fold {fold}: treino={len(tr)} validações internas={len(splits_abs)} teste={len(te)}")
    for tipo in tipos:
        for sem in SEMENTES:
            t0 = av.agora(); candidatos = {}
            # Cada participante do treino externo recebe previsão interna out-of-fold
            # uma vez por partição escolhida; densidade é escolhida pela AUC média.
            for dens in DENSIDADES:
                aucs, epocas_cv, y_cv, p_cv = [], [], [], []
                for j, (itr, iva) in enumerate(splits_abs):
                    torch.manual_seed(sem * 10000 + fold * 100 + j)
                    FEATS_CV = normalizar_atributos(itr)  # estatísticas internas ajustadas somente no treino interno
                    modelo_cv = RedeGrafo(tipo, N)
                    modelo_cv, auc_val, ep = treinar_g(modelo_cv, FEATS_CV, dens, d['y'], itr, iva,
                                                       sem * 10000 + fold * 100 + j)
                    p_inner = prever_g(modelo_cv, FEATS_CV, dens, iva)
                    aucs.append(float(auc_val)); epocas_cv.append(int(ep))
                    y_cv.append(d['y'][iva]); p_cv.append(p_inner)
                candidatos[dens] = dict(auc=float(np.mean(aucs)), auc_dp=float(np.std(aucs)),
                                         epocas=int(np.median(epocas_cv)), y=np.concatenate(y_cv),
                                         p=np.concatenate(p_cv), aucs=aucs, epocas_cv=epocas_cv)
            dens_esc = max(candidatos, key=lambda k: (candidatos[k]['auc'], -k))
            sel = candidatos[dens_esc]
            # Limiar derivado exclusivamente de previsões internas out-of-fold.
            lim = limiar_interno(sel['y'], sel['p'])
            # Refita no treino externo completo pelo número mediano de épocas escolhido internamente.
            torch.manual_seed(sem * 10000 + fold * 100 + 99)
            modelo = RedeGrafo(tipo, N).to(DEVICE)
            epocas_anteriores = EPOCAS_FIXAS
            EPOCAS_FIXAS = max(1, sel['epocas'])
            try:
                modelo, _, ep_final = treinar_g(modelo, FEATS, dens_esc, d['y'], tr, None,
                                                sem * 10000 + fold * 100 + 99)
            finally:
                EPOCAS_FIXAS = epocas_anteriores
            p = prever_g(modelo, FEATS, dens_esc, te); pr = (p >= lim).astype(np.int8)
            lin = av.linha_resultado(NOME_BASE, tipo, fold, sem, d['y'][te], pr, p, limiar=lim,
                 densidade=dens_esc, auc_val_interna=sel['auc'], auc_val_dp=sel['auc_dp'],
                 epocas=ep_final, k_interno=int(info['k_usado']), n_splits_internos=len(splits_abs),
                 seg_total=round(av.agora() - t0, 1))
            linhas.append(lin)
            escolhas.append(dict(fold=fold, modelo=tipo, semente=sem, densidade_escolhida=dens_esc,
                limiar=lim, auc_val_media=sel['auc'], auc_val_dp=sel['auc_dp'],
                epocas_refit=ep_final, **{f'auc_val_{k:.0%}': v['auc'] for k,v in candidatos.items()}))
            if sem == SEMENTES[0]:
                proba_oof[tipo][te], pred_oof[tipo][te] = p, pr
                torch.save(dict(estado=modelo.state_dict(), densidade=dens_esc, limiar=lim,
                    tipo=tipo, fold=int(fold), seed=int(sem), epocas=ep_final,
                    splits_internos=info), PASTA_MOD / 'grafos_folds' / f'{tipo.lower()}_fold{fold}_seed{sem}.pt')
            alerta = ' <-- ALERTA: uma única classe prevista' if lin['classe_unica_prevista'] else ''
            print(f"  {tipo:9s} seed={sem} dens={dens_esc:.0%} AUC={lin['auc_roc']:.3f} "
                  f"BalAcc={lin['balanced_accuracy']:.3f} AUCval={sel['auc']:.3f} limiar={lim:.2f} "
                  f"ep_refit={ep_final} ({lin['seg_total']:.0f}s){alerta}")
    pd.DataFrame(linhas).to_csv(PASTA_RES / '04_grafos_resultados_long.csv', index=False)
    pd.DataFrame(escolhas).to_csv(PASTA_RES / '04_grafos_escolha_densidade.csv', index=False)

for m in tipos:
    if np.any(pred_oof[m] < 0) or not np.isfinite(proba_oof[m]).all():
        raise RuntimeError(f'Previsões OOF incompletas para {m}; verifique folds sem validação interna viável.')
df_long = pd.DataFrame(linhas); df_long['tipo'] = 'externo'


FOLDS EXTERNOS


,fold,n_treino,n_teste,controles_teste,casos_teste,controles_treino,casos_treino,sites_teste,sobreposicao_sites,duas_classes_no_teste
0,0,830,41,19,22,449,381,YALE,0,True
1,1,698,173,101,72,367,331,"KKI, OLIN, SBL, UM_1",0,True
2,2,706,165,91,74,377,329,"CALTECH, LEUVEN_1, LEUVEN_2, PITT, TRINITY",0,True
3,3,547,324,175,149,293,254,"CMU, NYU, OHSU, SDSU, STANFORD, UCLA_1",0,True
4,4,703,168,82,86,386,317,"MAX_MUN, UCLA_2, UM_2, USM",0,True



=== Fold 0: treino=830 validações internas=3 teste=41
  GCN       seed=42 dens=10% AUC=0.641 BalAcc=0.555 AUCval=0.695 limiar=0.45 ep_refit=32 (461s)
  GCN       seed=43 dens=10% AUC=0.677 BalAcc=0.567 AUCval=0.683 limiar=0.39 ep_refit=28 (382s)
  GCN       seed=44 dens=10% AUC=0.660 BalAcc=0.657 AUCval=0.689 limiar=0.55 ep_refit=24 (367s)
  GAT       seed=42 dens=20% AUC=0.718 BalAcc=0.581 AUCval=0.673 limiar=0.48 ep_refit=32 (3202s)
  GAT       seed=43 dens=10% AUC=0.663 BalAcc=0.634 AUCval=0.669 limiar=0.47 ep_refit=37 (2799s)
  GAT       seed=44 dens=20% AUC=0.677 BalAcc=0.671 AUCval=0.684 limiar=0.69 ep_refit=29 (2657s)
  BrainGNN  seed=42 dens=20% AUC=0.732 BalAcc=0.676 AUCval=0.690 limiar=0.43 ep_refit=24 (583s)
  BrainGNN  seed=43 dens=20% AUC=0.670 BalAcc=0.522 AUCval=0.692 limiar=0.46 ep_refit=35 (617s)
  BrainGNN  seed=44 dens=20% AUC=0.622 BalAcc=0.638 AUCval=0.695 limiar=0.59 ep_refit=39 (622s)

=== Fold 1: treino=698 validações internas=3 teste=173
  GCN       seed=42 de

In [4]:
# 5. Resumo e salvamento
res = av.resumir_resultados(df_long, d, proba_oof)
pd.set_option('display.width', 200)
print('MÉDIA ± DP ENTRE FOLDS'); display(res['por_fold'])
print('\nMÉDIA PONDERADA PELO TAMANHO DO FOLD'); display(res['ponderado'])
print('\nAUC OOF (1ª semente) com IC 95% por bootstrap de sites'); display(res['auc_oof_ic'])
print('\nDENSIDADE ESCOLHIDA NA VALIDAÇÃO INTERNA (contagem por modelo)')
display(df_long.groupby(['modelo', 'densidade']).size().unstack(fill_value=0))
print('Colapso (uma classe prevista) em', int(df_long['classe_unica_prevista'].sum()), 'de', len(df_long), 'execuções.')
print('Critério a priori: IC inferior > 0,5 => "acima do acaso" (não é prova). A atenção do GAT não é explicação biológica.')

df_long.to_csv(PASTA_RES / '04_grafos_resultados_long.csv', index=False)
pd.DataFrame(escolhas).to_csv(PASTA_RES / '04_grafos_escolha_densidade.csv', index=False)
res['por_fold'].to_csv(PASTA_RES / '04_grafos_resumo_por_fold.csv')
res['ponderado'].to_csv(PASTA_RES / '04_grafos_resumo_ponderado.csv', index=False)
res['auc_oof_ic'].to_csv(PASTA_RES / '04_grafos_auc_oof_ic.csv', index=False)
av.salvar_oof(PASTA_RES / '04_grafos_predicoes_oof.npz', d, proba_oof, pred_oof)
print('Salvo em:', PASTA_RES)

MÉDIA ± DP ENTRE FOLDS


accuracy         balanced_accuracy         precision          recall         specificity              f1         auc_roc        
             mean     std              mean     std      mean     std    mean     std        mean     std    mean     std    mean     std
modelo                                                                                                                                   
BrainGNN   0.5979  0.0162            0.5968  0.0142    0.5729  0.0597  0.5997  0.1015      0.5939  0.0846  0.5786  0.0648  0.6378  0.0232
GAT        0.5813  0.0283            0.5863  0.0287    0.5259  0.1052  0.5864  0.1140      0.5863  0.1013  0.5475  0.0918  0.6336  0.0342
GCN        0.6026  0.0192            0.5999  0.0200    0.5838  0.0519  0.6075  0.1180      0.5923  0.1028  0.5829  0.0716  0.6460  0.0192


MÉDIA PONDERADA PELO TAMANHO DO FOLD


,modelo,accuracy_pond,balanced_accuracy_pond,precision_pond,recall_pond,specificity_pond,f1_pond,auc_roc_pond
0,BrainGNN,0.5943,0.5944,0.5597,0.5872,0.6015,0.5666,0.6306
1,GAT,0.5756,0.5813,0.5047,0.6029,0.5597,0.5455,0.6238
2,GCN,0.6016,0.5991,0.5774,0.5742,0.6240,0.5630,0.6408



AUC OOF (1ª semente) com IC 95% por bootstrap de sites


,modelo,auc,ic_inf,ic_sup,n_boot_validos,obs
0,GCN,0.6001,0.5694,0.6322,2000,
1,GAT,0.6061,0.5699,0.6440,2000,
2,BrainGNN,0.6128,0.5814,0.6447,2000,



DENSIDADE ESCOLHIDA NA VALIDAÇÃO INTERNA (contagem por modelo)


densidade,0.1,0.2
modelo,,
BrainGNN,4,11
GAT,7,8
GCN,13,2


Colapso (uma classe prevista) em 1 de 45 execuções.
Critério a priori: IC inferior > 0,5 => "acima do acaso" (não é prova). A atenção do GAT não é explicação biológica.
Salvo em: /home/maria/TCC_TEA/experimentos/08_grafos_tea/Results
